In [1]:
import numpy as np
import random



In [2]:
# -----------------------------
# Environment setup
# -----------------------------
GRID_SIZE = 5
START_STATE = (0, 0)
GOAL_STATE = (4, 4)

ACTIONS = {
    0: (-1, 0),  # up
    1: (1, 0),   # down
    2: (0, -1),  # left
    3: (0, 1)    # right
}

NUM_ACTIONS = len(ACTIONS)



In [3]:
# -----------------------------
# Hyperparameters
# -----------------------------
ALPHA = 0.1      # learning rate
GAMMA = 0.9      # discount factor
EPSILON = 0.1    # exploration rate
EPISODES = 500



In [4]:
# -----------------------------
# Helper functions
# -----------------------------
def step(state, action):
    x, y = state
    dx, dy = ACTIONS[action]

    nx = min(max(x + dx, 0), GRID_SIZE - 1)
    ny = min(max(y + dy, 0), GRID_SIZE - 1)

    next_state = (nx, ny)
    reward = 10 if next_state == GOAL_STATE else -1

    return next_state, reward

def choose_action(Q, state):
    if random.random() < EPSILON:
        return random.randint(0, NUM_ACTIONS - 1)
    return np.argmax(Q[state[0], state[1]])



In [5]:
# -----------------------------
# Initialize Q-tables
# -----------------------------
Q_qlearning = np.zeros((GRID_SIZE, GRID_SIZE, NUM_ACTIONS))
Q_sarsa = np.zeros((GRID_SIZE, GRID_SIZE, NUM_ACTIONS))



In [6]:
# -----------------------------
# Q-Learning (off-policy)
# -----------------------------
for _ in range(EPISODES):
    state = START_STATE

    while state != GOAL_STATE:
        action = choose_action(Q_qlearning, state)
        next_state, reward = step(state, action)

        best_next = np.max(Q_qlearning[next_state[0], next_state[1]])
        Q_qlearning[state[0], state[1], action] += ALPHA * (
            reward + GAMMA * best_next - Q_qlearning[state[0], state[1], action]
        )

        state = next_state



In [7]:
# -----------------------------
# SARSA (on-policy)
# -----------------------------
for _ in range(EPISODES):
    state = START_STATE
    action = choose_action(Q_sarsa, state)

    while state != GOAL_STATE:
        next_state, reward = step(state, action)
        next_action = choose_action(Q_sarsa, next_state)

        Q_sarsa[state[0], state[1], action] += ALPHA * (
            reward + GAMMA * Q_sarsa[next_state[0], next_state[1], next_action]
            - Q_sarsa[state[0], state[1], action]
        )

        state, action = next_state, next_action



In [8]:
# -----------------------------
# Extract learned policies
# -----------------------------
def extract_policy(Q):
    policy = np.zeros((GRID_SIZE, GRID_SIZE), dtype=int)
    for i in range(GRID_SIZE):
        for j in range(GRID_SIZE):
            policy[i, j] = np.argmax(Q[i, j])
    return policy

policy_qlearning = extract_policy(Q_qlearning)
policy_sarsa = extract_policy(Q_sarsa)



In [9]:
# -----------------------------
# Results
# -----------------------------
print("Q-Learning Policy (action indices):")
print(policy_qlearning)

print("\nSARSA Policy (action indices):")
print(policy_sarsa)


Q-Learning Policy (action indices):
[[3 1 1 0 0]
 [3 1 1 1 1]
 [3 3 1 1 1]
 [3 3 3 1 1]
 [3 3 3 3 0]]

SARSA Policy (action indices):
[[3 1 1 1 1]
 [1 1 1 1 1]
 [3 1 1 1 1]
 [3 3 3 1 1]
 [0 3 3 3 0]]


**Q-Learning and SARSA were implemented and evaluated in the same 5×5 grid-world environment. The learned policies demonstrate clear differences in behavior due to the algorithms’ learning strategies. Q-Learning, an off-policy method, learned a more direct and aggressive path toward the goal by consistently selecting actions with the highest estimated future reward, even when those actions were rarely taken during exploration. In contrast, SARSA, an on-policy method, learned a more conservative policy that reflects the agent’s exploratory behavior, resulting in safer and more uniform action choices across the grid. These differences arise because Q-Learning updates its value function using the greedy next action, whereas SARSA updates based on the action actually taken, leading to distinct policy structures despite identical environments and reward functions.**